# 09 — Diagnóstico de la clasificación española con BART

Analiza `scam_es_FINAL_classified.csv` (clasificación BART). NO reclasifica nada.

## Anomalías a investigar
1. `insurance` (170, 13.8%) parece inflada: sus ejemplos son "estafa del hijo en apuros".
2. `payment_app` (2) y `tech_support` (2) sospechosamente bajos.


In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 120)

df = pd.read_csv("../data/raw/scam_es_FINAL_classified.csv")
print(f"Tweets: {len(df)}")


## Confianza por categoría

In [ ]:
conf = df.groupby("predicted_category").agg(
    n=("confidence_score","count"),
    conf_media=("confidence_score","mean"),
    conf_mediana=("confidence_score","median"),
).sort_values("n", ascending=False).round(3)
print(conf)


## insurance — ¿de qué query vienen?

In [ ]:
ins = df[df["predicted_category"]=="insurance"]
print(f"Total insurance: {len(ins)}\n")
if "query_labels" in df.columns:
    print("Query de origen de los tweets en 'insurance':")
    for q in ["phishing","payment_apps","crypto","romance","impersonation"]:
        n = ins["query_labels"].fillna("").str.contains(q).sum()
        pct = n/len(ins)*100 if len(ins) else 0
        print(f"  {q:<16} {n:>5}  ({pct:.1f}%)")


## insurance — ¿cuántos mencionan 'hijo en apuros'?

In [ ]:
mask_hijo = ins["text"].fillna("").str.contains("hijo en apuros|hija en apuros|he cambiado de num|cambiado de número", case=False, regex=True)
print(f"Tweets de insurance que mencionan 'hijo en apuros' o similar: {mask_hijo.sum()} de {len(ins)}")
print(f"  ({mask_hijo.sum()/len(ins)*100:.1f}% de la categoria insurance)")
print()
# Cuantos mencionan realmente seguros
mask_seguro = ins["text"].fillna("").str.contains("seguro|póliza|poliza|aseguradora", case=False, regex=True)
print(f"Tweets de insurance que mencionan realmente 'seguro/póliza': {mask_seguro.sum()} de {len(ins)}")


## insurance — muestra de 25 ejemplos

In [ ]:
print("=== 25 TWEETS CLASIFICADOS COMO insurance ===\n")
for _, r in ins.nlargest(25,"confidence_score").iterrows():
    q = r.get("query_labels","?")
    print(f"[conf {r['confidence_score']:.2f}] [query: {q}]")
    print(f"  {str(r['text'])[:200]}")
    print()


## payment_app y tech_support — ¿qué pasó?

In [ ]:
for cat in ["payment_app","tech_support"]:
    sub = df[df["predicted_category"]==cat]
    print(f"=== {cat}: {len(sub)} tweets ===")
    for _, r in sub.iterrows():
        print(f"  [{r['confidence_score']:.2f}] {str(r['text'])[:180]}")
    print()

# A donde fueron los tweets de la query payment_apps?
if "query_labels" in df.columns:
    pay = df[df["query_labels"].fillna("").str.contains("payment_apps")]
    print(f"Tweets cuya query era 'payment_apps': {len(pay)}")
    print("Se clasificaron como:")
    print(pay["predicted_category"].value_counts())


## Resumen

In [ ]:
ins_hijo = df[(df["predicted_category"]=="insurance")]["text"].fillna("").str.contains("hijo en apuros|hija en apuros|cambiado de num", case=False, regex=True).sum()
print("="*55)
print("  RESUMEN DEL DIAGNÓSTICO")
print("="*55)
print(f"insurance: {(df['predicted_category']=='insurance').sum()} tweets")
print(f"  de los cuales ~{ins_hijo} son 'estafa del hijo en apuros'")
print(f"  (eso NO es fraude de seguros -> mal clasificado)")
print()
print(f"payment_app: {(df['predicted_category']=='payment_app').sum()} tweets")
print(f"tech_support: {(df['predicted_category']=='tech_support').sum()} tweets")
print("="*55)
